# Thí nghiệm 03 — Chuẩn bị dữ liệu Financial

## 1. Đọc và kiểm tra dữ liệu gốc

### Mục tiêu

Notebook chuẩn bị dữ liệu tài chính cho thí nghiệm dự báo chuỗi thời gian của DeepKoopFormer.

Dữ liệu nguồn là `crypto_coins.csv`, bao gồm thông tin giá và giao dịch của nhiều loại tiền mã hóa.

Thực nghiệm Financial cuối cùng **chỉ sử dụng Bitcoin (BTC)**. Dữ liệu BTC được chia thành hai đoạn thời gian liên tiếp, mỗi đoạn 2.900 quan sát.

### Quy trình

1. Kiểm tra dữ liệu CSV.
2. Xác định hai segment của BTC.
3. Ghép hai segment thành ma trận hai kênh.
4. Chia Train/Test theo thời gian, tỷ lệ 80/20.
5. Chuẩn hóa Min-Max bằng scaler chỉ được khớp trên Train.
6. Tạo sliding windows.
7. Kiểm tra bundle dữ liệu và SHA-256.

Các tệp dữ liệu đã được tạo từ trước. Trong lần dọn notebook này, chế độ ghi dữ liệu được tắt để bảo toàn kết quả huấn luyện.

In [1]:
from pathlib import Path

import hashlib
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import MinMaxScaler


# 1. Tìm thư mục gốc repository
project_root = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / "experiments" / "03_financial"
            / "data" / "crypto_coins.csv"
        ).is_file()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy experiments/03_financial/data/crypto_coins.csv"
    )


# 2. Đường dẫn Financial
experiment_dir = project_root / "experiments" / "03_financial"
data_dir = experiment_dir / "data"

DATA_PATH = data_dir / "crypto_coins.csv"


# 3. Chế độ an toàn
# Không thay đổi thành True trong quá trình cleanup
REBUILD_DATA_FILES = False


# 4. Đọc dữ liệu gốc
df = pd.read_csv(DATA_PATH)

required_columns = {
    "CloseTime",
    "ClosePrice",
    "Symbol",
    "Name"
}

assert required_columns.issubset(df.columns)
assert df.shape == (27585, 10)


# 5. Kiểm tra dữ liệu
print("=== FINANCIAL DATASET ===")
print("Source:", DATA_PATH)
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum().to_string())

print("\nFirst 5 rows:")
display(df.head())

=== FINANCIAL DATASET ===
Source: c:\Users\Win 11 New\Desktop\pa gk\experiments\03_financial\data\crypto_coins.csv
Shape: (27585, 10)

Columns:
['CloseTime', 'OpenPrice', 'HighPrice', 'LowPrice', 'ClosePrice', 'volume', 'MarketCap', 'Symbol', 'Name', 'Date']

Missing values:
CloseTime     0
OpenPrice     0
HighPrice     0
LowPrice      0
ClosePrice    0
volume        0
MarketCap     0
Symbol        0
Name          0
Date          0

First 5 rows:


,CloseTime,OpenPrice,HighPrice,LowPrice,ClosePrice,volume,MarketCap,Symbol,Name,Date
0,2018-09-01 00:00:00,7042.95,7101.03,6998.87,7017.35,1353.6716,9549478.0,BTC,Bitcoin,2018-09-01
1,2018-09-01 06:00:00,7015.78,7070.00,7015.22,7042.34,338.5957,2384278.0,BTC,Bitcoin,2018-09-01
2,2018-09-01 12:00:00,7035.04,7131.85,7024.51,7058.97,874.0480,6180810.5,BTC,Bitcoin,2018-09-01
3,2018-09-01 18:00:00,7058.97,7230.00,7021.29,7196.27,1570.7167,11247725.0,BTC,Bitcoin,2018-09-01
4,2018-09-02 00:00:00,7193.30,7300.18,7169.99,7185.01,1807.0952,13054075.0,BTC,Bitcoin,2018-09-02


## 2. Khảo sát các đồng tiền trong dữ liệu nguồn

Tệp CSV chứa nhiều loại tiền mã hóa. Trước khi lựa chọn chuỗi BTC, cần kiểm tra số bản ghi, khoảng thời gian và cách sắp xếp các đồng tiền trong dữ liệu gốc.

Thao tác này nhằm xác minh 5.800 bản ghi đầu tiên thuộc BTC và có thể được sử dụng để tạo hai segment.

Không sử dụng ETH như một kênh dự báo trong thực nghiệm cuối cùng.

In [3]:
# 1. Chuyển thời gian về datetime
df["CloseTime"] = pd.to_datetime(df["CloseTime"])


# 2. Tổng hợp thông tin từng đồng tiền
coin_summary = (
    df.groupby(["Symbol", "Name"], sort=False)
    .agg(
        Records=("ClosePrice", "size"),
        StartDate=("CloseTime", "min"),
        EndDate=("CloseTime", "max")
    )
    .reset_index()
)


# 3. Kiểm tra thứ tự các đồng tiền trong CSV
transitions = df.loc[
    df["Symbol"].ne(df["Symbol"].shift()),
    ["Symbol", "CloseTime"]
]


print("=== SOURCE DATA AUDIT ===")
print("Number of cryptocurrencies:", df["Symbol"].nunique())

print("\nRecords by cryptocurrency:")
display(coin_summary)

print("\nSymbol transitions:")
display(transitions)

print("\nFirst 5800 rows are BTC:",
      df["Symbol"].iloc[:5800].eq("BTC").all())

=== SOURCE DATA AUDIT ===
Number of cryptocurrencies: 10

Records by cryptocurrency:


,Symbol,Name,Records,StartDate,EndDate
0,BTC,Bitcoin,5863,2018-09-01 00:00:00,2022-09-05 12:00:00
1,ETH,Ethereum,5863,2018-09-01 00:00:00,2022-09-05 12:00:00
2,ADA,Cardano,1138,2021-11-24 18:00:00,2022-09-05 12:00:00
3,XRP,Ripple,5863,2018-09-01 00:00:00,2022-09-05 12:00:00
4,USDT,Tether,1787,2021-06-15 18:00:00,2022-09-05 12:00:00
5,MATIC,Polygon,1565,2021-08-10 12:00:00,2022-09-05 12:00:00
6,USDC,USD Coin,2735,2020-10-21 18:00:00,2022-09-05 12:00:00
7,SHIB,SHIBA INU,1,2021-12-09 18:00:00,2021-12-09 18:00:00
8,FTM,Fantom,881,2022-01-13 18:00:00,2022-09-05 12:00:00
9,UMA,Universal Market Access,1889,2021-04-22 18:00:00,2022-09-05 06:00:00



Symbol transitions:


,Symbol,CloseTime
0,BTC,2018-09-01 00:00:00
5863,ETH,2018-09-01 00:00:00
11726,ADA,2021-11-24 18:00:00
12864,XRP,2018-09-01 00:00:00
18727,USDT,2021-06-15 18:00:00
20514,MATIC,2021-08-10 12:00:00
22079,USDC,2020-10-21 18:00:00
24814,SHIB,2021-12-09 18:00:00
24815,FTM,2022-01-13 18:00:00
25696,UMA,2021-04-22 18:00:00



First 5800 rows are BTC: True



## 3. Lựa chọn hai segment của Bitcoin

### 3.1. Phương pháp lựa chọn

Thực nghiệm sử dụng 5.800 giá trị `ClosePrice` đầu tiên của BTC trong thứ tự lưu trữ của CSV gốc.

Dữ liệu được chia thành hai đoạn liên tiếp:

- **BTC Segment 1:** 2.900 quan sát đầu tiên.
- **BTC Segment 2:** 2.900 quan sát tiếp theo.

Mỗi segment có tần suất lấy mẫu 6 giờ.

### 3.2. Khoảng thời gian

| Segment | Bắt đầu | Kết thúc |
|---|---|---|
| Segment 1 | 01/09/2018 00:00 | 25/08/2020 18:00 |
| Segment 2 | 26/08/2020 00:00 | 20/08/2022 18:00 |

Hai segment được ghép theo chỉ số vị trí tương ứng, không phải theo timestamp trùng nhau.

### 3.3. Lưu ý

Đây là hai đoạn của cùng một đồng tiền BTC, không phải hai chuỗi BTC và ETH.

Giữ nguyên thứ tự các bản ghi nguồn để duy trì tính nhất quán với dữ liệu đã tạo và checkpoint đã huấn luyện.

In [5]:
# 1. Lấy chính xác 5800 bản ghi đầu tiên của CSV
btc_selected = df.iloc[:5800].copy()

assert len(btc_selected) == 5800
assert btc_selected["Symbol"].eq("BTC").all()


# 2. Chia thành hai segment liên tiếp
btc_segment_1 = btc_selected.iloc[:2900].copy()
btc_segment_2 = btc_selected.iloc[2900:5800].copy()

assert len(btc_segment_1) == 2900
assert len(btc_segment_2) == 2900


# 3. Kiểm tra thứ tự thời gian
assert btc_segment_1["CloseTime"].is_monotonic_increasing
assert btc_segment_2["CloseTime"].is_monotonic_increasing

assert not btc_segment_1["CloseTime"].duplicated().any()
assert not btc_segment_2["CloseTime"].duplicated().any()


# 4. Kiểm tra khoảng lấy mẫu 6 giờ
expected_interval = pd.Timedelta(hours=6)

for name, segment in [
    ("BTC Segment 1", btc_segment_1),
    ("BTC Segment 2", btc_segment_2),
]:
    intervals = segment["CloseTime"].diff().dropna()

    assert intervals.eq(expected_interval).all(), (
        f"Time interval mismatch: {name}"
    )

    print(f"\n{name}")
    print("Shape:", segment.shape)
    print("Start:", segment["CloseTime"].iloc[0])
    print("End:", segment["CloseTime"].iloc[-1])
    print("Sampling interval:", expected_interval)


# 5. Kiểm tra hai đoạn liên tiếp
assert (
    btc_segment_1["CloseTime"].iloc[-1]
    + expected_interval
    == btc_segment_2["CloseTime"].iloc[0]
)

print("\nBoth BTC segments validated.")


BTC Segment 1
Shape: (2900, 10)
Start: 2018-09-01 00:00:00
End: 2020-08-25 18:00:00
Sampling interval: 0 days 06:00:00

BTC Segment 2
Shape: (2900, 10)
Start: 2020-08-26 00:00:00
End: 2022-08-20 18:00:00
Sampling interval: 0 days 06:00:00

Both BTC segments validated.



## 4. Xây dựng chuỗi thời gian hai kênh

Hai segment BTC được ghép theo chiều cột:

\[
X =
\begin{bmatrix}
s_{1,1} & s_{2,1}\\
s_{1,2} & s_{2,2}\\
\vdots & \vdots\\
s_{1,2900} & s_{2,2900}
\end{bmatrix}
\]

Trong đó \(s_1\) và \(s_2\) là hai segment của BTC.

Kích thước dữ liệu cuối cùng:

\[
X\in\mathbb{R}^{2900\times2}
\]

Sử dụng kiểu dữ liệu `float32`.

Dữ liệu `crypto_coins.npy` đã tồn tại. Cell chỉ đối chiếu với tệp này, không ghi đè.


In [6]:
# 1. Lấy ClosePrice từ dữ liệu nguồn
close_prices = df["ClosePrice"].to_numpy(
    dtype=np.float32
)

selected = close_prices[:5800]

assert len(selected) == 5800


# 2. Ghép hai đoạn BTC thành hai kênh
financial_data = np.column_stack([
    selected[:2900],
    selected[2900:5800]
]).astype(np.float32)

assert financial_data.shape == (2900, 2)


# 3. Kiểm tra phép ghép khớp với hai segment
np.testing.assert_array_equal(
    financial_data[:, 0],
    btc_segment_1["ClosePrice"].to_numpy(dtype=np.float32)
)

np.testing.assert_array_equal(
    financial_data[:, 1],
    btc_segment_2["ClosePrice"].to_numpy(dtype=np.float32)
)


# 4. Bảo vệ file NPY hiện có
npy_path = data_dir / "crypto_coins.npy"

if REBUILD_DATA_FILES:
    if npy_path.exists():
        raise FileExistsError(
            f"Không được ghi đè dữ liệu đã có: {npy_path}"
        )

    np.save(npy_path, financial_data)

else:
    if not npy_path.is_file():
        raise FileNotFoundError(npy_path)

    stored_data = np.load(npy_path)

    np.testing.assert_array_equal(
        stored_data,
        financial_data
    )

    print("Existing NPY matches selected BTC segments.")


# 5. Tổng kết
print("\n=== FINANCIAL DATA ===")
print("Shape:", financial_data.shape)
print("Dtype:", financial_data.dtype)
print("Channels:", ["BTC_segment_1", "BTC_segment_2"])
print("First 5 rows:")
print(financial_data[:5])

Existing NPY matches selected BTC segments.

=== FINANCIAL DATA ===
Shape: (2900, 2)
Dtype: float32
Channels: ['BTC_segment_1', 'BTC_segment_2']
First 5 rows:
[[ 7017.35 11328.88]
 [ 7042.34 11367.  ]
 [ 7058.97 11383.9 ]
 [ 7196.27 11456.31]
 [ 7185.01 11469.3 ]]



## 5. Chia Train/Test và chuẩn hóa Min-Max

### 5.1. Chia dữ liệu

Dữ liệu được chia tuần tự theo thời gian:

- Train: 80% đầu tiên, 2.320 quan sát.
- Test: 20% cuối cùng, 580 quan sát.

Không xáo trộn dữ liệu.

### 5.2. Chuẩn hóa

Sử dụng `MinMaxScaler(feature_range=(0, 1))`.

Scaler chỉ được khớp trên Train và sau đó được sử dụng để biến đổi cả Train và Test.

Điều này tránh sử dụng thống kê từ Test trong quá trình chuẩn hóa Train.

Mỗi cột tương ứng với một segment BTC và được chuẩn hóa bằng thống kê của chính cột đó trên Train.

### 5.3. Kích thước

| Tập | Shape |
|---|---|
| Train | (2320, 2) |
| Test | (580, 2) |

Giá trị Test không bắt buộc nằm trong khoảng [0, 1] vì scaler không được khớp lại trên Test.


In [7]:
# 1. Chia tuần tự theo thời gian
split_idx = int(len(financial_data) * 0.8)

train_raw = financial_data[:split_idx]
test_raw = financial_data[split_idx:]

assert train_raw.shape == (2320, 2)
assert test_raw.shape == (580, 2)


# 2. Fit scaler chỉ trên Train
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_raw)


# 3. Biến đổi Train và Test
train_scaled = scaler.transform(train_raw).astype(np.float32)
test_scaled = scaler.transform(test_raw).astype(np.float32)


# 4. Kiểm tra kết quả
assert train_scaled.shape == (2320, 2)
assert test_scaled.shape == (580, 2)

assert np.isfinite(train_scaled).all()
assert np.isfinite(test_scaled).all()


print("=== FINANCIAL TRAIN / TEST ===")
print("Original:", financial_data.shape)
print("Train:", train_scaled.shape)
print("Test:", test_scaled.shape)

print("\nTrain min:", train_scaled.min(axis=0))
print("Train max:", train_scaled.max(axis=0))

print("\nTest min:", test_scaled.min(axis=0))
print("Test max:", test_scaled.max(axis=0))


# 5. Khoảng thời gian của từng segment
for name, segment in [
    ("BTC Segment 1", btc_segment_1),
    ("BTC Segment 2", btc_segment_2),
]:
    print(f"\n{name}")

    print(
        "Train:",
        segment["CloseTime"].iloc[0],
        "->",
        segment["CloseTime"].iloc[split_idx - 1]
    )

    print(
        "Test:",
        segment["CloseTime"].iloc[split_idx],
        "->",
        segment["CloseTime"].iloc[-1]
    )

=== FINANCIAL TRAIN / TEST ===
Original: (2900, 2)
Train: (2320, 2)
Test: (580, 2)

Train min: [0. 0.]
Train max: [1. 1.]

Test min: [0.3367909  0.13852938]
Test max: [0.89590466 0.64723927]

BTC Segment 1
Train: 2018-09-01 00:00:00 -> 2020-04-02 18:00:00
Test: 2020-04-03 00:00:00 -> 2020-08-25 18:00:00

BTC Segment 2
Train: 2020-08-26 00:00:00 -> 2022-03-28 18:00:00
Test: 2022-03-29 00:00:00 -> 2022-08-20 18:00:00



## 6. Tạo sliding windows

### 6.1. Đầu vào và nhãn

Với độ dài đầu vào \(P\) và chân trời dự báo \(H\):

- Input: \(P\) thời điểm liên tiếp của hai kênh BTC.
- Target: \(H\) thời điểm tương lai của hai kênh.

Kích thước một mẫu đầu vào:

\[
X_i\in\mathbb{R}^{P\times2}
\]

Kích thước nhãn sau khi làm phẳng:

\[
y_i\in\mathbb{R}^{2H}
\]

Nhãn được làm phẳng theo **channel-major order**: toàn bộ \(H\) giá trị của kênh thứ nhất, rồi đến toàn bộ \(H\) giá trị của kênh thứ hai.

### 6.2. Quy tắc tạo cửa sổ

Để giữ nguyên thực nghiệm, sử dụng:

`range(len(data) - seq_len - horizon)`

Không thêm `+1` vào giới hạn vòng lặp.

### 6.3. Cấu hình minh họa

Với `P = 40`, `H = 16`:

- `X_train`: (2264, 40, 2).
- `y_train`: (2264, 32).
- `X_test`: (524, 40, 2).
- `y_test`: (524, 32).

Đây là phép kiểm tra cách tạo cửa sổ, không thực hiện huấn luyện.


In [8]:
def create_windows(data, seq_len, horizon):
    """
    Tạo sliding windows cho chuỗi thời gian đa biến.

    Input:  (N, seq_len, channels)
    Target: (N, channels * horizon)

    Giữ nguyên quy tắc tạo cửa sổ của thực nghiệm gốc.
    """
    X, y = [], []

    for i in range(len(data) - seq_len - horizon):
        x_window = data[i:i + seq_len]

        y_window = data[
            i + seq_len:i + seq_len + horizon
        ].T.reshape(-1)

        X.append(x_window)
        y.append(y_window)

    X = torch.tensor(np.stack(X), dtype=torch.float32)
    y = torch.tensor(np.stack(y), dtype=torch.float32)

    return X, y


# 1. Cấu hình minh họa
p = 40
H = 16


# 2. Tạo cửa sổ Train và Test
X_train, y_train = create_windows(
    train_scaled, p, H
)

X_test, y_test = create_windows(
    test_scaled, p, H
)


# 3. Kiểm tra kích thước
assert X_train.shape == (2264, 40, 2)
assert y_train.shape == (2264, 32)

assert X_test.shape == (524, 40, 2)
assert y_test.shape == (524, 32)


print("=== FINANCIAL SLIDING WINDOWS ===")
print("Patch length:", p)
print("Horizon:", H)

print("\nTrain:")
print("X:", tuple(X_train.shape))
print("y:", tuple(y_train.shape))

print("\nTest:")
print("X:", tuple(X_test.shape))
print("y:", tuple(y_test.shape))

=== FINANCIAL SLIDING WINDOWS ===
Patch length: 40
Horizon: 16

Train:
X: (2264, 40, 2)
y: (2264, 32)

Test:
X: (524, 40, 2)
y: (524, 32)



## 7. Đóng gói dữ liệu và xác minh tính toàn vẹn

### 7.1. Bundle dữ liệu

Tệp `financial_train_test.pt` chứa:

- Tensor Train và Test.
- Tên hai kênh BTC Segment 1 và BTC Segment 2.
- Số kênh đầu vào.
- Tỷ lệ Train/Test.
- Số quan sát của từng tập.
- Giá trị min/max của scaler.
- SHA-256 fingerprint của Train và Test.

### 7.2. SHA-256

Fingerprint được tính từ byte của hai tensor `float32`, theo thứ tự Train rồi Test.

Fingerprint của dữ liệu đang sử dụng:

`9383a90ba5109d569c2f965b9698120f893a86f146efd05f7585659a3913f519`

Đây cũng là fingerprint đã được đối chiếu thành công với 224 checkpoint Financial.

### 7.3. Bảo toàn kết quả

Trong chế độ cleanup, notebook không ghi đè tệp `.pt`.

Cell chỉ kiểm tra rằng dữ liệu tính được từ pipeline vẫn khớp hoàn toàn với bundle đã lưu.

Không huấn luyện lại mô hình hoặc tạo lại checkpoint.


In [9]:
# 1. Chuyển dữ liệu chuẩn hóa thành tensor CPU
train_tensor = torch.from_numpy(
    train_scaled.copy()
).float()

test_tensor = torch.from_numpy(
    test_scaled.copy()
).float()


# 2. Tính SHA-256 theo đúng quy trình cũ
data_hash = hashlib.sha256(
    train_tensor.numpy().tobytes()
    + test_tensor.numpy().tobytes()
).hexdigest()


# 3. Fingerprint đã được xác minh với 224 checkpoint
EXPECTED_HASH = (
    "9383a90ba5109d569c2f965b9698120f893a86"
    "f146efd05f7585659a3913f519"
)

assert data_hash == EXPECTED_HASH, (
    "Dữ liệu đã thay đổi so với phiên bản huấn luyện!"
)


# 4. Đóng gói metadata (trong bộ nhớ)
bundle = {
    "train": train_tensor,
    "test": test_tensor,
    "channels": [
        "BTC_segment_1",
        "BTC_segment_2"
    ],
    "input_dim": 2,
    "train_ratio": 0.8,
    "train_size": len(train_tensor),
    "test_size": len(test_tensor),
    "scaler_data_min": scaler.data_min_.copy(),
    "scaler_data_max": scaler.data_max_.copy(),
    "data_hash": data_hash,
}


# 5. Đường dẫn bundle đã lưu
output_path = data_dir / "financial_train_test.pt"


# 6. Chỉ cho phép tạo mới nếu người dùng chủ động bật
if REBUILD_DATA_FILES:
    if output_path.exists():
        raise FileExistsError(
            f"Không được ghi đè bundle đã có: {output_path}"
        )

    torch.save(bundle, output_path)


# 7. Đọc bundle hiện có để xác minh
if not output_path.is_file():
    raise FileNotFoundError(output_path)

# File thuộc chính repository, chứa cả metadata NumPy.
saved = torch.load(
    output_path,
    map_location="cpu",
    weights_only=False
)


# 8. Kiểm tra fingerprint và metadata
assert saved["data_hash"] == EXPECTED_HASH

assert saved["channels"] == [
    "BTC_segment_1",
    "BTC_segment_2"
]

assert saved["input_dim"] == 2
assert saved["train_size"] == 2320
assert saved["test_size"] == 580


# 9. Kiểm tra dữ liệu tensor giống hoàn toàn
assert torch.equal(
    saved["train"],
    train_tensor
)

assert torch.equal(
    saved["test"],
    test_tensor
)


# 10. Kiểm tra metadata của scaler
np.testing.assert_array_equal(
    saved["scaler_data_min"],
    scaler.data_min_
)

np.testing.assert_array_equal(
    saved["scaler_data_max"],
    scaler.data_max_
)


# 11. Tổng kết
print("=== FINANCIAL DATA VALIDATION ===")

print("Bundle:", output_path)
print("Train:", tuple(saved["train"].shape))
print("Test:", tuple(saved["test"].shape))
print("Channels:", saved["channels"])

print("\nSHA256:", saved["data_hash"])
print("Expected hash matched:", True)

print("\nExisting bundle matches preprocessing pipeline.")
print("No checkpoint or dataset file overwritten.")

=== FINANCIAL DATA VALIDATION ===
Bundle: c:\Users\Win 11 New\Desktop\pa gk\experiments\03_financial\data\financial_train_test.pt
Train: (2320, 2)
Test: (580, 2)
Channels: ['BTC_segment_1', 'BTC_segment_2']

SHA256: 9383a90ba5109d569c2f965b9698120f893a86f146efd05f7585659a3913f519
Expected hash matched: True

Existing bundle matches preprocessing pipeline.
No checkpoint or dataset file overwritten.
